# Demo APIs

Este notebook muestra el funcionamiento separado de:
- Neo4j (usuarios y contactos)
- Redis (colas de mensajes)
- MongoDB (histórico y snapshots)


In [20]:
from api import create_mongo_repository, create_neo4j_repository, create_redis_repository

## Neo4j

In [ ]:
# !!! cambiar la contraseña por la propia !!!
repo = create_neo4j_repository(auth=("neo4j", "neo4j"))
repo.clear_database()

In [22]:
alice = repo.create_node("User", {
    "name": "Alice",
    "phone": "123456789"
})

bob = repo.create_node("User", {
    "name": "Bob",
    "phone": "987654321"
})

rosa = repo.create_node("User", {
    "name": "Rosa",
    "phone": "999999999"
})

alice, bob, rosa

({'phone': '123456789',
  'name': 'Alice',
  'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:17',
  'node_type': 'User'},
 {'phone': '987654321',
  'name': 'Bob',
  'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:19',
  'node_type': 'User'},
 {'phone': '999999999',
  'name': 'Rosa',
  'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:25',
  'node_type': 'User'})

In [23]:
repo.read_nodes("User")

[{'phone': '123456789',
  'name': 'Alice',
  'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:17',
  'node_type': 'User'},
 {'phone': '987654321',
  'name': 'Bob',
  'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:19',
  'node_type': 'User'},
 {'phone': '999999999',
  'name': 'Rosa',
  'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:25',
  'node_type': 'User'}]

In [24]:
repo.read_node_by_id("User", alice["id"])

{'phone': '123456789',
 'name': 'Alice',
 'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:17',
 'node_type': 'User'}

In [25]:
repo.create_node_unique_constraint("User", "phone")

In [26]:
repo.create_relation(
    from_label="User",
    from_props={"phone": "123456789"},
    to_label="User",
    to_props={"phone": "987654321"},
    rel_type="AreConnected",
    rel_props={"message_count": 0}
)

[{'message_count': 0,
  'id': '5:9e9cd09d-672f-4a30-988d-9d13b764e2fa:6',
  'type': 'AreConnected'}]

In [27]:
repo.create_relation_by_id(
    from_label="User",
    from_id=bob["id"],
    to_label="User",
    to_id=rosa["id"],
    rel_type="AreConnected",
    rel_props={"message_count": 0}
)

{'message_count': 0,
 'id': '5:9e9cd09d-672f-4a30-988d-9d13b764e2fa:7',
 'type': 'AreConnected'}

In [28]:
repo.read_relations(relation_type="AreConnected")

[{'message_count': 0,
  'id': '5:9e9cd09d-672f-4a30-988d-9d13b764e2fa:6',
  'type': 'AreConnected',
  'from_node': {'phone': '123456789',
   'name': 'Alice',
   'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:17'},
  'to_node': {'phone': '987654321',
   'name': 'Bob',
   'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:19'}},
 {'message_count': 0,
  'id': '5:9e9cd09d-672f-4a30-988d-9d13b764e2fa:7',
  'type': 'AreConnected',
  'from_node': {'phone': '987654321',
   'name': 'Bob',
   'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:19'},
  'to_node': {'phone': '999999999',
   'name': 'Rosa',
   'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:25'}}]

In [29]:
repo.get_neighbors("User", bob["id"])

[{'phone': '999999999',
  'name': 'Rosa',
  'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:25'},
 {'phone': '123456789',
  'name': 'Alice',
  'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:17'}]

In [30]:
repo.is_connected(
    "User",
    "User",
    "AreConnected",
    alice["id"],
    bob["id"]
)

True

In [31]:
repo._increment_property(
    property_name="message_count",
    node_A_label="User",
    node_B_label="User",
    node_A_id=alice["id"],
    node_B_id=bob["id"],
    rel_type="AreConnected"
)

1

In [32]:
repo.read_relations(
    relation_type="AreConnected",
    from_props={"phone": "123456789"},
    to_props={"phone": "987654321"}
)

[{'message_count': 1,
  'id': '5:9e9cd09d-672f-4a30-988d-9d13b764e2fa:6',
  'type': 'AreConnected',
  'from_node': {'phone': '123456789',
   'name': 'Alice',
   'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:17'},
  'to_node': {'phone': '987654321',
   'name': 'Bob',
   'id': '4:9e9cd09d-672f-4a30-988d-9d13b764e2fa:19'}}]

In [33]:
repo.delete_nodes("User", {"phone": "999999999"})

1

In [34]:
repo.close()

## Redis
Redis devuelve en bytes todas las queries, luego en service es donde hago los decodes.

In [35]:
repo = create_redis_repository()
repo.clear_database()

In [36]:
repo.keys("*")

[]

In [37]:
repo.insert_hash("user:1", "name", "Alice")
repo.insert_hash("user:1", "phone", "123456789")

repo.insert_hash("user:2", "name", "Bob")
repo.insert_hash("user:2", "phone", "987654321")

In [38]:
repo.find_hash("user:1", "name")

b'Alice'

In [39]:
repo.find_hash_all("user:1")

{b'name': b'Alice', b'phone': b'123456789'}

In [40]:
repo.update_hash("user:1", "name", "Alice Cooper")

True

In [41]:
repo.update_hash("user:1", "email", "alice@mail.com")

False

In [42]:
repo.delete_hash("user:1", "phone")

True

In [43]:
repo.find_hash_all("user:1")

{b'name': b'Alice Cooper'}

In [44]:
repo.push_list("messages", "hola")
repo.push_list("messages", "que tal")
repo.push_list("messages", "adios")

In [45]:
repo.read_list("messages")

[b'hola', b'que tal', b'adios']

In [46]:
repo.pop_list("messages", timeout=1)

'hola'

In [47]:
repo.delete_from_list("messages", "adios")

1

In [48]:
repo.read_list("messages")

[b'que tal']

In [49]:
repo.keys("*")

['user:1', 'messages', 'user:2']

In [50]:
repo.clear_database()
repo.keys("*")

[]

In [51]:
repo.close()

## Mongo

In [52]:
repo = create_mongo_repository()
repo.clear_collection("messages")

In [53]:
msg1_id = repo.insert_one(
    "messages",
    {
        "from": "alice",
        "to": "bob",
        "content": "Hola Bob",
        "timestamp": 1700000000
    }
)

msg2_id = repo.insert_one(
    "messages",
    {
        "from": "bob",
        "to": "alice",
        "content": "Hola Alice",
        "timestamp": 1700000010
    }
)

msg1_id, msg2_id

('69640fee9e6737aaad5e2e38', '69640fee9e6737aaad5e2e39')

In [54]:
repo.find_one(
    "messages",
    {"from": "alice"}
)

{'_id': ObjectId('69640fee9e6737aaad5e2e38'),
 'from': 'alice',
 'to': 'bob',
 'content': 'Hola Bob',
 'timestamp': 1700000000}

In [55]:
repo.find_many(
    "messages",
    {"to": "alice"}
)

[{'_id': ObjectId('69640fee9e6737aaad5e2e39'),
  'from': 'bob',
  'to': 'alice',
  'content': 'Hola Alice',
  'timestamp': 1700000010}]

In [56]:
repo.find_many("messages")

[{'_id': ObjectId('69640fee9e6737aaad5e2e38'),
  'from': 'alice',
  'to': 'bob',
  'content': 'Hola Bob',
  'timestamp': 1700000000},
 {'_id': ObjectId('69640fee9e6737aaad5e2e39'),
  'from': 'bob',
  'to': 'alice',
  'content': 'Hola Alice',
  'timestamp': 1700000010}]

In [57]:
repo.find_many(
    "messages",
    sort=[("timestamp", 1)]
)

[{'_id': ObjectId('69640fee9e6737aaad5e2e38'),
  'from': 'alice',
  'to': 'bob',
  'content': 'Hola Bob',
  'timestamp': 1700000000},
 {'_id': ObjectId('69640fee9e6737aaad5e2e39'),
  'from': 'bob',
  'to': 'alice',
  'content': 'Hola Alice',
  'timestamp': 1700000010}]

In [58]:
repo.find_one_sorted(
    "messages",
    filter={},
    sort=[("timestamp", -1)]
)

{'_id': ObjectId('69640fee9e6737aaad5e2e39'),
 'from': 'bob',
 'to': 'alice',
 'content': 'Hola Alice',
 'timestamp': 1700000010}

In [59]:
repo.update_one(
    "messages",
    {"from": "alice"},
    {"content": "Hola Bob!!!"}
)

True

In [60]:
repo.find_one(
    "messages",
    {"from": "alice"}
)

{'_id': ObjectId('69640fee9e6737aaad5e2e38'),
 'from': 'alice',
 'to': 'bob',
 'content': 'Hola Bob!!!',
 'timestamp': 1700000000}

In [61]:
repo.delete_one(
    "messages",
    {"from": "bob"}
)

True

In [62]:
repo.find_many("messages")

[{'_id': ObjectId('69640fee9e6737aaad5e2e38'),
  'from': 'alice',
  'to': 'bob',
  'content': 'Hola Bob!!!',
  'timestamp': 1700000000}]

In [63]:
repo.delete_many(
    "messages",
    {"from": "alice"}
)

1

In [64]:
repo.find_many("messages")

[]

In [65]:
repo.clear_collection("messages")
repo.find_many("messages")

[]

In [66]:
repo.close()